# Type 3 transitions: merging tissue borders

A **type 3 transition** (Fletcher et al. 2013) joins two free borders: when a
border vertex comes closer than `settings["t3_threshold"]` to a free edge of
another cell, the vertex is merged into that edge, which is split in two. The
two cells then share a junction.

In tyssue 2.0:

- `topology.sheet_topology.t3_transition` performs one merge,
  `behaviors.sheet.actions.t3_transitions` finds and performs every merge,
  and the event `behaviors.sheet.t3` runs it at each time step.
- `EulerSolver(..., with_t3=True)` adds that event to the solver. This is for
  `Sheet` objects only.
- The removal of small triangular faces, which older versions called a
  "type 3" transition, is now the **type 2 transition**: see the end of this
  notebook.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython import display

from tyssue import History, PlanarGeometry as geom, Sheet
from tyssue.config.geometry import planar_spec
from tyssue.dynamics import effectors, model_factory
from tyssue.draw import create_gif, sheet_view
from tyssue.solvers.viscous import EulerSolver

## Two patches of cells

We build two hexagonal patches side by side, separated by a gap. Their borders
are free edges: they belong to a single face.

In [ ]:
def two_patches(gap):
    patch = Sheet.planar_sheet_2d("patch", 5, 5, 1, 1)
    patch.sanitize(trim_borders=True, order_edges=True)
    width = patch.vert_df["x"].max() - patch.vert_df["x"].min()
    datasets = {}
    for name in ("vert", "edge", "face"):
        left = patch.datasets[name].copy()
        right = patch.datasets[name].copy()
        right.index = right.index + len(left)
        if name in ("vert", "face"):
            right["x"] += width + gap
        if name == "edge":
            right[["srce", "trgt"]] += patch.Nv
            right["face"] += patch.Nf
        datasets[name] = pd.concat([left, right])
    sheet = Sheet("two", datasets, specs=planar_spec(), coords=["x", "y"])
    sheet.reset_index()
    sheet.reset_topo()
    geom.update_all(sheet)
    return sheet


sheet = two_patches(gap=0.4)
sheet.face_df["patch"] = (sheet.face_df.index >= sheet.Nf // 2).astype(int)

fig, ax = sheet_view(sheet, face={"visible": True, "color": sheet.face_df["patch"], "colormap": "coolwarm"})

## Driving the patches together

The model has line tension, area elasticity and an `ActiveMigration` force.
`ActiveMigration` reads each cell's direction from the face columns `mx`, `my`
(and `mz` in 3D) and splits the force `migration_strength` between the cell's
vertices. The left patch migrates to the right, the right patch to the left.

`settings["t3_threshold"]` sets the merge distance. We also set
`settings["detach_boundary"] = False` (see below).

In [ ]:
model = model_factory([
    effectors.LineTension,
    effectors.FaceAreaElasticity,
    effectors.ActiveMigration,
])
sheet.update_specs(model.specs, reset=True)

sheet.face_df["prefered_area"] = sheet.face_df["area"].mean()
sheet.edge_df["line_tension"] = 0.05
sheet.vert_df["viscosity"] = 1.0

sheet.face_df["mx"] = np.where(sheet.face_df["patch"] == 0, 1.0, -1.0)
sheet.face_df["my"] = 0.0
sheet.face_df["migration_strength"] = 0.2

sheet.settings["t3_threshold"] = 0.05
sheet.settings["detach_boundary"] = False
geom.update_all(sheet)


def junctions_between_patches(sheet):
    shared = sheet.edge_df[sheet.edge_df["opposite"] >= 0]
    this_patch = sheet.face_df.loc[shared["face"], "patch"].to_numpy()
    other_face = sheet.edge_df.loc[shared["opposite"], "face"]
    other_patch = sheet.face_df.loc[other_face, "patch"].to_numpy()
    return int((this_patch != other_patch).sum() // 2)


print("junctions between the patches:", junctions_between_patches(sheet))

In [ ]:
history = History(sheet)
solver = EulerSolver(sheet, geom, model, history=history, with_t3=True)
solver.solve(tf=4, dt=0.05)

print("junctions between the patches:", junctions_between_patches(sheet))
print("valid mesh:", sheet.validate())

Once the patches are joined, we stop the migration and let the tissue relax.
`find_intersections` checks that no face folded or overlapped on the way.

In [ ]:
sheet.face_df["migration_strength"] = 0.0
solver.solve(tf=10, dt=0.05)

print("intersection free:", sheet.find_intersections().clean)
fig, ax = sheet_view(sheet, face={"visible": True, "color": sheet.face_df["patch"], "colormap": "coolwarm"})

In [ ]:
create_gif(history, "t3_fusion.gif", num_frames=30, face={"visible": True, "color": lambda s: s.face_df["patch"], "colormap": "coolwarm"})
display.Image("t3_fusion.gif")

## Keeping the new junctions: `detach_boundary`

A vertex created by a T3 sits on the tissue border with four neighbours.
During rearrangements, `reconnect` (and the `detach_vertices` action) detaches
such rank 4 border vertices by default, which would undo the merge.
`settings["detach_boundary"] = False` keeps them. We force a detachment on two
copies of the merged tissue to compare:

In [ ]:
from tyssue.behaviors.sheet.actions import detach_vertices

for detach in (True, False):
    trial = sheet.copy()
    # probabilities high enough that every candidate vertex is detached
    trial.settings.update(sheet.settings, detach_boundary=detach, p_4=1e9, p_5p=1e9, dt=1.0)
    geom.update_all(trial)
    detach_vertices(trial)
    print(f"detach_boundary={detach}: {sheet.Nv} -> {trial.Nv} vertices")

## Type 2 transitions (formerly "type 3")

Removing a triangular face that has become too small is the **type 2
transition**. The options were renamed accordingly:

| tyssue < 2.0 | tyssue 2.0 |
| --- | --- |
| `QSSolver(with_t3=True)` | `QSSolver(with_t2=True)` |
| `topology.auto_t3` | `topology.auto_t2` |
| `all_rearangements(..., with_t3=)` | `all_rearangements(..., with_t2=)` |
| `single_rearangement(..., with_t3=)` | `single_rearangement(..., with_t2=)` |

The old names still work and raise a `FutureWarning`. Note that `with_t3` on
`EulerSolver` is the new type 3 transition shown above, not the old alias.

In [ ]:
import warnings

from tyssue.solvers import QSSolver

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    QSSolver(with_t1=False, with_t3=True)
print(caught[0].category.__name__, ":", caught[0].message)